# Baseline v1 — full validation evaluation

Notebook evaluates the saved `best.pt` on a reconstructed historical validation set of 896 images.

The primary metrics are calculated using the total TP, FP, and FN for the entire validation (micro aggregation), rather than as an average of individual image percentages. Additionally, a threshold sweep, class-aware NMS, per-class metric calculation, and standard Ultralytics validation for mAP are performed.

The competition FAR is the false-discovery proportion among predictions:

- `FAR = FP / (TP + FP) = 1 - Precision`.

> This is the reconstructed training-time validation baseline v1. One duplicate FSC label is removed during reads, as Ultralytics did in the historical run.

In [10]:
from collections import Counter, defaultdict
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from torchvision.ops import batched_nms
from tqdm.auto import tqdm
from ultralytics import RTDETR

import os

os.chdir(ROOT)

print("Current working directory:", Path.cwd())

ROOT = Path.cwd().resolve()
if ROOT.name == 'baseline_v1':
    ROOT = ROOT.parents[1]
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent

print('Project root:', ROOT)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Current working directory: /home/irdorat/ML_comp
Project root: /home/irdorat/ML_comp
PyTorch: 2.11.0+cu128
CUDA available: True


## Settings

`MIN_CONFIDENCE` is used for a single inference pass. The saved predictions are then filtered for all `THRESHOLDS` values ​​without rerunning the model.

In [2]:
WEIGHTS = ROOT / 'runs/baseline_v1/weights/best.pt'
VAL_MANIFEST = ROOT / 'data/splits/baseline_v1/val.txt'
DATA_CONFIG = ROOT / 'configs/baseline_v1/dataset.yaml'

IMAGE_SIZE = 640
BATCH_SIZE = 8
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
MIN_CONFIDENCE = 0.05
MATCH_IOU = 0.50
NMS_IOU = 0.50
OPERATING_CONFIDENCE = 0.50
THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)

assert WEIGHTS.exists(), f'Checkpoint not found: {WEIGHTS}'
assert VAL_MANIFEST.exists(), f'Validation manifest not found: {VAL_MANIFEST}'
assert DATA_CONFIG.exists(), f'Dataset config not found: {DATA_CONFIG}'

print('Weights:', WEIGHTS)
print('Validation manifest:', VAL_MANIFEST)
print('Device:', DEVICE)
print('Minimum inference confidence:', MIN_CONFIDENCE)
print('Matching IoU:', MATCH_IOU)
print('NMS IoU:', NMS_IOU)

Weights: /home/irdorat/ML_comp/runs/baseline_v1/weights/best.pt
Validation manifest: /home/irdorat/ML_comp/data/splits/baseline_v1/val.txt
Device: 0
Minimum inference confidence: 0.05
Matching IoU: 0.5
NMS IoU: 0.5


## Init validation and ground truth

In [3]:
def resolve_manifest(manifest_path):
    paths = []
    for raw_line in manifest_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('./'):
            path = (manifest_path.parent / line[2:]).resolve()
        else:
            path = Path(line)
            if not path.is_absolute():
                path = (ROOT / path).resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        paths.append(path)
    return paths


def image_to_label_path(image_path):
    parts = list(image_path.parts)
    images_index = parts.index('images')
    parts[images_index] = 'labels'
    return Path(*parts).with_suffix('.txt')


def load_yolo_ground_truth(label_path, image_width, image_height):
    boxes, classes = [], []
    seen = set()
    duplicates = 0

    for line in label_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        values = line.split()
        class_id = int(values[0])
        xc, yc, width, height = map(float, values[1:5])
        key = (class_id, xc, yc, width, height)
        if key in seen:
            duplicates += 1
            continue
        seen.add(key)
        boxes.append([
            (xc - width / 2) * image_width,
            (yc - height / 2) * image_height,
            (xc + width / 2) * image_width,
            (yc + height / 2) * image_height,
        ])
        classes.append(class_id)

    return (
        np.asarray(boxes, dtype=np.float32).reshape(-1, 4),
        np.asarray(classes, dtype=np.int64),
        duplicates,
    )


val_paths = resolve_manifest(VAL_MANIFEST)
assert len(val_paths) == 896, f'Expected 896 validation images, found {len(val_paths)}'
print('Validation images:', len(val_paths))

Validation images: 896


## One inference pass across all 896 images

Predictions are stored in memory only. Images and results are not written to disk.

In [4]:
model = RTDETR(str(WEIGHTS))
samples = []
duplicate_labels_removed = 0
timing_rows = []
wall_start = time.perf_counter()

for start in tqdm(range(0, len(val_paths), BATCH_SIZE), desc='Inference batches'):
    batch_paths = val_paths[start:start + BATCH_SIZE]
    batch_results = model.predict(
        source=[str(path) for path in batch_paths],
        conf=MIN_CONFIDENCE,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        save=False,
        verbose=False,
    )

    for image_path, result in zip(batch_paths, batch_results):
        image_height, image_width = result.orig_shape
        label_path = image_to_label_path(image_path)
        gt_boxes, gt_classes, duplicates = load_yolo_ground_truth(
            label_path, image_width, image_height
        )
        duplicate_labels_removed += duplicates

        if result.boxes is None or len(result.boxes) == 0:
            pred_boxes = np.empty((0, 4), dtype=np.float32)
            pred_classes = np.empty((0,), dtype=np.int64)
            pred_conf = np.empty((0,), dtype=np.float32)
        else:
            pred_boxes = result.boxes.xyxy.cpu().numpy().astype(np.float32)
            pred_classes = result.boxes.cls.int().cpu().numpy().astype(np.int64)
            pred_conf = result.boxes.conf.cpu().numpy().astype(np.float32)

        samples.append({
            'image': image_path,
            'gt_boxes': gt_boxes,
            'gt_classes': gt_classes,
            'pred_boxes': pred_boxes,
            'pred_classes': pred_classes,
            'pred_conf': pred_conf,
        })
        timing_rows.append(result.speed.copy())

wall_time_seconds = time.perf_counter() - wall_start
class_names = batch_results[0].names

print('Processed images:', len(samples))
print('Ground-truth objects:', sum(len(sample['gt_boxes']) for sample in samples))
print('Duplicate labels removed:', duplicate_labels_removed)
print(f'Wall time: {wall_time_seconds:.2f} s')

Inference batches: 100%|██████████| 112/112 [00:32<00:00,  3.41it/s]

Processed images: 896
Ground-truth objects: 4294
Duplicate labels removed: 1
Wall time: 32.89 s


## Comparison of detections and calculation of metrics

In [5]:
def box_iou_matrix(boxes_a, boxes_b):
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    top_left = np.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])
    bottom_right = np.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])
    intersection_wh = np.clip(bottom_right - top_left, 0, None)
    intersection = intersection_wh[..., 0] * intersection_wh[..., 1]
    area_a = np.clip(boxes_a[:, 2] - boxes_a[:, 0], 0, None) * np.clip(boxes_a[:, 3] - boxes_a[:, 1], 0, None)
    area_b = np.clip(boxes_b[:, 2] - boxes_b[:, 0], 0, None) * np.clip(boxes_b[:, 3] - boxes_b[:, 1], 0, None)
    union = area_a[:, None] + area_b[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)


def apply_class_aware_nms(boxes, classes, confidence, iou_threshold):
    if len(boxes) == 0:
        return boxes, classes, confidence
    keep = batched_nms(
        torch.as_tensor(boxes, dtype=torch.float32),
        torch.as_tensor(confidence, dtype=torch.float32),
        torch.as_tensor(classes, dtype=torch.int64),
        iou_threshold,
    ).cpu().numpy()
    return boxes[keep], classes[keep], confidence[keep]


def match_sample(gt_boxes, gt_classes, pred_boxes, pred_classes, pred_conf, iou_threshold):
    matched_gt = set()
    matched_pred = set()
    iou_matrix = box_iou_matrix(pred_boxes, gt_boxes)

    for pred_index in np.argsort(-pred_conf):
        candidates = [
            gt_index for gt_index in range(len(gt_boxes))
            if gt_index not in matched_gt and gt_classes[gt_index] == pred_classes[pred_index]
        ]
        if not candidates:
            continue
        best_gt = max(candidates, key=lambda gt_index: iou_matrix[pred_index, gt_index])
        if iou_matrix[pred_index, best_gt] >= iou_threshold:
            matched_gt.add(best_gt)
            matched_pred.add(int(pred_index))

    return matched_pred, matched_gt


def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def aggregate_metrics(samples, threshold, use_nms, return_per_class=False):
    totals = Counter()
    per_class = defaultdict(Counter)

    for sample in samples:
        mask = sample['pred_conf'] >= threshold
        pred_boxes = sample['pred_boxes'][mask]
        pred_classes = sample['pred_classes'][mask]
        pred_conf = sample['pred_conf'][mask]

        if use_nms:
            pred_boxes, pred_classes, pred_conf = apply_class_aware_nms(
                pred_boxes, pred_classes, pred_conf, NMS_IOU
            )

        matched_pred, matched_gt = match_sample(
            sample['gt_boxes'], sample['gt_classes'],
            pred_boxes, pred_classes, pred_conf, MATCH_IOU
        )

        tp = len(matched_pred)
        fp = len(pred_boxes) - tp
        fn = len(sample['gt_boxes']) - len(matched_gt)
        totals.update(tp=tp, fp=fp, fn=fn, predictions=len(pred_boxes), gt=len(sample['gt_boxes']))

        if return_per_class:
            for pred_index, class_id in enumerate(pred_classes):
                per_class[int(class_id)]['tp' if pred_index in matched_pred else 'fp'] += 1
            for gt_index, class_id in enumerate(sample['gt_classes']):
                if gt_index not in matched_gt:
                    per_class[int(class_id)]['fn'] += 1
                per_class[int(class_id)]['gt'] += 1

    precision = safe_divide(totals['tp'], totals['tp'] + totals['fp'])
    recall = safe_divide(totals['tp'], totals['tp'] + totals['fn'])
    row = {
        'threshold': threshold, 'nms': use_nms,
        'gt': totals['gt'], 'predictions': totals['predictions'],
        'tp': totals['tp'], 'fp': totals['fp'], 'fn': totals['fn'],
        'precision': precision, 'recall': recall,
        'f1': safe_divide(2 * precision * recall, precision + recall),
        'far': safe_divide(totals['fp'], totals['tp'] + totals['fp']),
    }
    return (row, per_class) if return_per_class else row

## Threshold Sweep on the Full Validation Set

In [6]:
sweep_rows = []
for threshold in tqdm(THRESHOLDS, desc='Threshold sweep'):
    sweep_rows.append(aggregate_metrics(samples, float(threshold), use_nms=False))
    sweep_rows.append(aggregate_metrics(samples, float(threshold), use_nms=True))

sweep_df = pd.DataFrame(sweep_rows)
sweep_df['recall_pass'] = sweep_df['recall'] >= 0.85
sweep_df['far_pass'] = sweep_df['far'] <= 0.20
sweep_df['all_quality_targets_pass'] = sweep_df['recall_pass'] & sweep_df['far_pass']

display(sweep_df.style.format({
    'threshold': '{:.2f}', 'precision': '{:.3f}', 'recall': '{:.3f}',
    'f1': '{:.3f}', 'far': '{:.3f}',
}))

Threshold sweep: 100%|██████████| 19/19 [00:03<00:00,  5.40it/s]


,threshold,nms,gt,predictions,tp,fp,fn,precision,recall,f1,far_pred,far_gt,recall_pass,far_pred_pass,far_gt_pass,all_quality_targets_pass
0,0.05,False,4294,74592,4157,70435,137,0.056,0.968,0.105,0.944,16.403,True,False,False,False
1,0.05,True,4294,48131,4141,43990,153,0.086,0.964,0.158,0.914,10.245,True,False,False,False
2,0.10,False,4294,23888,4121,19767,173,0.173,0.960,0.292,0.827,4.603,True,False,False,False
3,0.10,True,4294,17822,4108,13714,186,0.231,0.957,0.371,0.769,3.194,True,False,False,False
4,0.15,False,4294,12027,4099,7928,195,0.341,0.955,0.502,0.659,1.846,True,False,False,False
5,0.15,True,4294,10253,4087,6166,207,0.399,0.952,0.562,0.601,1.436,True,False,False,False
6,0.20,False,4294,8440,4082,4358,212,0.484,0.951,0.641,0.516,1.015,True,False,False,False
7,0.20,True,4294,7684,4069,3615,225,0.530,0.948,0.679,0.470,0.842,True,False,False,False
8,0.25,False,4294,6871,4051,2820,243,0.590,0.943,0.726,0.410,0.657,True,False,False,False
9,0.25,True,4294,6422,4040,2382,254,0.629,0.941,0.754,0.371,0.555,True,False,False,False


## Operating point confidence=0.5 and metrics by class

In [7]:
raw_operating, _ = aggregate_metrics(
    samples, OPERATING_CONFIDENCE, use_nms=False, return_per_class=True
)
nms_operating, nms_per_class = aggregate_metrics(
    samples, OPERATING_CONFIDENCE, use_nms=True, return_per_class=True
)
operating_df = pd.DataFrame([raw_operating, nms_operating])
display(operating_df.style.format({
    'threshold': '{:.2f}', 'precision': '{:.3f}', 'recall': '{:.3f}',
    'f1': '{:.3f}', 'far': '{:.3f}',
}))

per_class_rows = []
for class_id in range(25):
    counts = nms_per_class[class_id]
    precision = safe_divide(counts['tp'], counts['tp'] + counts['fp'])
    recall = safe_divide(counts['tp'], counts['tp'] + counts['fn'])
    per_class_rows.append({
        'class_id': class_id, 'class': class_names[class_id],
        'gt': counts['gt'], 'tp': counts['tp'], 'fp': counts['fp'], 'fn': counts['fn'],
        'precision': precision, 'recall': recall,
        'far': safe_divide(counts['fp'], counts['tp'] + counts['fp']),
    })

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df.style.format({
    'precision': '{:.3f}', 'recall': '{:.3f}',
    'far': '{:.3f}',
}))

,threshold,nms,gt,predictions,tp,fp,fn,precision,recall,f1,far_pred,far_gt
0,0.50,False,4294,4822,3889,933,405,0.807,0.906,0.853,0.193,0.217
1,0.50,True,4294,4706,3879,827,415,0.824,0.903,0.862,0.176,0.193


,class_id,class,gt,tp,fp,fn,precision,recall,far_pred,far_gt
0,0,HM,3,0,0,3,0.000,0.000,0.000,0.000
1,1,LQS,5,0,0,5,0.000,0.000,0.000,0.000
2,2,QHS,141,99,57,42,0.635,0.702,0.365,0.404
3,3,MS,372,319,98,53,0.765,0.858,0.235,0.263
4,4,A1_SU-35,273,246,33,27,0.882,0.901,0.118,0.121
5,5,A2_C-130,337,331,17,6,0.951,0.982,0.049,0.050
6,6,A3_C-17,236,231,47,5,0.831,0.979,0.169,0.199
7,7,A4_C-5,100,96,36,4,0.727,0.960,0.273,0.360
8,8,A5_F-16,205,196,41,9,0.827,0.956,0.173,0.200
9,9,A6_TU-160,71,65,13,6,0.833,0.915,0.167,0.183


## Average processing time for the provided crop

This is not a benchmark image of 10,000×10,000.

In [8]:
timing_df = pd.DataFrame(timing_rows)
timing_summary = pd.DataFrame({
    'stage': ['preprocess', 'inference', 'postprocess', 'model stages total', 'wall time per image'],
    'mean_ms': [
        timing_df['preprocess'].mean(), timing_df['inference'].mean(),
        timing_df['postprocess'].mean(), timing_df[['preprocess', 'inference', 'postprocess']].sum(axis=1).mean(),
        wall_time_seconds * 1000 / len(samples),
    ],
    'median_ms': [
        timing_df['preprocess'].median(), timing_df['inference'].median(),
        timing_df['postprocess'].median(), timing_df[['preprocess', 'inference', 'postprocess']].sum(axis=1).median(),
        np.nan,
    ],
})
timing_summary

,stage,mean_ms,median_ms
0,preprocess,1.750903,1.660995
1,inference,21.889402,21.889928
2,postprocess,0.642940,0.564265
3,model stages total,24.283245,24.180561
4,wall time per image,36.709351,NaN


## Standard Ultralytics validation: mAP

This cell performs the second inference pass and stores standard validation artifacts in `runs/baseline_v1/full_val`.

In [11]:
ultralytics_metrics = model.val(
    data=str(DATA_CONFIG),
    split='val',
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(ROOT / 'runs/baseline_v1'),
    name='full_val',
    exist_ok=True,
)

map_results = {
    'precision': float(ultralytics_metrics.box.p.mean()),
    'recall': float(ultralytics_metrics.box.r.mean()),
    'mAP50': float(ultralytics_metrics.box.map50),
    'mAP50-95': float(ultralytics_metrics.box.map),
}
pd.DataFrame([map_results])

Ultralytics 8.4.114 🚀 Python-3.13.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
rt-detr-l summary: 315 layers, 32,035,115 parameters, 0 gradients, 105.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1341.3±995.5 MB/s, size: 129.6 KB)
val: Scanning /home/irdorat/ML_comp/data/splits/baseline_v1/../../raw/labels/train... 896 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 896/896 768.8it/s 1.2s0.0s
val: /home/irdorat/ML_comp/data/splits/baseline_v1/../../raw/images/train/fsc_TG-N22.33-E120.62-lv20-Google_crop0002.jpg: 1 duplicate labels removed
val: New cache created: /home/irdorat/ML_comp/data/splits/baseline_v1/../../raw/labels/train.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 112/112 4.7it/s 24.1s0.2s
                   all        896       4294      0.774      0.795      0.801      0.449
                    HM          3          3          1          0     0.0996     0.07

,precision,recall,mAP50,mAP50-95
0,0.773553,0.79535,0.801153,0.449123


## Final report

In [14]:
passing_points = sweep_df[sweep_df["all_quality_targets_pass"]].copy()
best_passing = (
    passing_points.sort_values(["recall", "f1"], ascending=False).iloc[0]
    if len(passing_points)
    else None
)

selected = nms_operating

best_text = (
    f"Best passing operating point: confidence={best_passing['threshold']:.2f}, "
    f"NMS={bool(best_passing['nms'])}, "
    f"Recall={best_passing['recall']:.3f}, "
    f"FAR={best_passing['far']:.3f}."
    if best_passing is not None
    else (
        "None of the evaluated operating points simultaneously satisfied "
        "the Recall and FAR targets."
    )
)

summary = f"""
# Baseline v1 Results on the Full Validation Set

## Evaluation Protocol

- Validation manifest: `{VAL_MANIFEST.relative_to(ROOT)}`;
- Number of images: {len(samples)};
- Ground-truth objects after duplicate removal: {selected['gt']};
- Matching IoU threshold: {MATCH_IOU:.2f};
- Operating confidence threshold: {OPERATING_CONFIDENCE:.2f};
- Class-aware NMS IoU threshold: {NMS_IOU:.2f}.

## Aggregated Results at Confidence={OPERATING_CONFIDENCE:.2f} After NMS

| Metric | Result | Target |
|---|---:|---:|
| TP / FP / FN | {selected['tp']} / {selected['fp']} / {selected['fn']} | — |
| Precision | {selected['precision']:.3f} | — |
| Recall | {selected['recall']:.3f} | ≥ 0.85 |
| F1 score | {selected['f1']:.3f} | — |
| FAR | {selected['far']:.3f} | ≤ 0.20 |
| mAP50 | {map_results['mAP50']:.3f} | — |
| mAP50–95 | {map_results['mAP50-95']:.3f} | — |

## Operating Point Selection

{best_text}

## Processing Time

Mean model inference time per provided image crop:
{timing_df['inference'].mean():.2f} ms.

This measurement does not represent end-to-end processing time for a full
10,000 × 10,000-pixel scene.
"""

display(Markdown(summary))


# Baseline v1 Results on the Full Validation Set

## Evaluation Protocol

- Validation manifest: `data/splits/baseline_v1/val.txt`;
- Number of images: 896;
- Ground-truth objects after duplicate removal: 4294;
- Matching IoU threshold: 0.50;
- Operating confidence threshold: 0.50;
- Class-aware NMS IoU threshold: 0.50.

## Aggregated Results at Confidence=0.50 After NMS

| Metric | Result | Target |
|---|---:|---:|
| TP / FP / FN | 3879 / 827 / 415 | — |
| Precision | 0.824 | — |
| Recall | 0.903 | ≥ 0.85 |
| F1 score | 0.862 | — |
| FAR | 0.176 | ≤ 0.20 |
| mAP50 | 0.801 | — |
| mAP50–95 | 0.449 | — |

## Operating Point Selection

Best passing operating point: confidence=0.50, NMS=True, Recall=0.903, FAR=0.176.

## Processing Time

Mean model inference time per provided image crop:
21.89 ms.

This measurement does not represent end-to-end processing time for a full
10,000 × 10,000-pixel scene.
